# RTX-KG2 queries — minimal all-Soufflé pipeline

## Setup

In [1]:
import os
import json
import subprocess
import time
import pandas as pd
from dotenv import load_dotenv
import anthropic

pd.set_option("display.max_colwidth", 80)

NODES_TSV = "rtx-kg2_nodes_2.8.1.tsv"
EDGES_TSV = "rtx-kg2_edges_2.8.1.tsv"
PREDICATES_TSV = "distinct_core_predicate.tsv"
OUT_DIR = "souffle_out"

EDGE_HEADER = (
    ".decl edge(subj:symbol, pred:symbol, obj:symbol, neg:symbol)\n"
    '.input edge(IO=file, filename="' + EDGES_TSV + '", columns="16:0:17:3")'
)

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ["APIKEY"])
MODEL = "claude-sonnet-4-5"

def llm(system, user, max_tokens=1024):
    msg = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                 system=system, messages=[{"role": "user", "content": user}])
    return msg.content[0].text

def _json(text):
    i, j = text.find("{"), text.rfind("}")
    return json.loads(text[i:j + 1])

def _souffle_code(text):
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1].rsplit("```", 1)[0]
    return t.strip()

KG2_PREDICATES = (pd.read_csv(PREDICATES_TSV, sep="\t")
                  .sort_values("n", ascending=False)["predicate"].astype(str).tolist())
print(str(len(KG2_PREDICATES)) + " predicates")

61 predicates


## 1. Run Soufflé

In [2]:
def _profile_summary(prof_path, wall):
    if not os.path.exists(prof_path):
        return
    rel = json.load(open(prof_path))["root"]["program"]["relation"]
    compute = 0.0
    for name, info in sorted(rel.items()):
        rt = info.get("runtime", {})
        dur = (rt.get("end", 0) - rt.get("start", 0)) / 1e6
        compute += dur
        if dur >= 0.01:
            print("    compute " + name + ": " + format(dur, ".2f") + "s (" + str(info.get("num-tuples")) + " tuples)")
    print("    load+overhead: " + format(max(0.0, wall - compute), ".2f") + "s")

def run_souffle(program, out_name="result"):
    os.makedirs(OUT_DIR, exist_ok=True)
    dl_path = out_name + ".dl"
    with open(dl_path, "w") as f:
        f.write(program + "\n")
    prof = os.path.join(OUT_DIR, out_name + "_profile.json")
    cmd = "ulimit -s unlimited 2>/dev/null || ulimit -s 1048576; exec souffle " + dl_path + " -F . -D " + OUT_DIR + " -p " + prof
    t0 = time.perf_counter()
    subprocess.run(["bash", "-c", cmd], check=True, env=dict(os.environ, OMP_STACKSIZE="1G"))
    wall = time.perf_counter() - t0
    print("[souffle:" + out_name + "] wall " + format(wall, ".1f") + "s")
    _profile_summary(prof, wall)
    path = os.path.join(OUT_DIR, out_name + ".csv")
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return pd.DataFrame()
    return pd.read_csv(path, sep="\t", header=None, dtype=str).fillna("")

## 2. Retrieve node and predicate IDs

In [3]:
# --- NCATS name resolution + node normalization -------------------------------
# NameRes  (text  -> concept CURIEs)         replaces the Souffle match() search.
# NodeNorm (CURIE -> equivalent-ID clique)   replaces manual "all" selection.
import urllib.request, urllib.parse, urllib.error

NAMERES_URL = "https://name-resolution-sri.renci.org/lookup"
NODENORM_URL = "https://nodenormalization-sri.renci.org/get_normalized_nodes"
_nameres_cache, _nodenorm_cache = {}, {}

def _get_json(url, data=None, method="GET", timeout=40):
    req = urllib.request.Request(url, data=data, method=method,
                                 headers={"Accept": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)

def nameres_lookup(term, limit=10, biolink_type=None):
    key = (term, limit, biolink_type)
    if key in _nameres_cache:
        return _nameres_cache[key]
    params = {"string": term, "limit": limit}
    if biolink_type:
        params["biolink_type"] = biolink_type
    url = NAMERES_URL + "?" + urllib.parse.urlencode(params)
    try:
        res = _get_json(url)
    except urllib.error.HTTPError as e:
        if e.code != 405:
            raise
        res = _get_json(url, data=b"", method="POST")   # older deployments are POST-only
    out = []
    for x in res:
        types = x.get("types") or []
        out.append({"id": x.get("curie"), "name": x.get("label") or "",
                    "category": types[0] if types else "",
                    "score": round(x.get("score", 0) or 0, 1),
                    "clique_n": x.get("clique_identifier_count")})
    _nameres_cache[key] = out
    return out

# conflate merges gene<->protein cliques; drug_chemical_conflate merges drug<->chemical
# (e.g. metformin the chemical with metformin the drug). Both boost recall; set
# DRUG_CHEMICAL_CONFLATE=False if you want tighter, less aggressive drug cliques.
DRUG_CHEMICAL_CONFLATE = True

def nodenorm_clique(curie):
    if curie in _nodenorm_cache:
        return _nodenorm_cache[curie]
    params = {"curie": curie, "conflate": "true",
              "drug_chemical_conflate": "true" if DRUG_CHEMICAL_CONFLATE else "false"}
    node = {}
    try:
        data = _get_json(NODENORM_URL + "?" + urllib.parse.urlencode(params))
        node = data.get(curie) or {}
    except Exception as e:
        print("  [nodenorm error] " + repr(e))
    eqids = [e.get("identifier") for e in node.get("equivalent_identifiers", []) if e.get("identifier")]
    eqids = list(dict.fromkeys(eqids)) or [curie]   # de-dup, preserve order
    res = {"preferred": (node.get("id") or {}).get("identifier", curie),
           "equivalent_ids": eqids,
           "categories": node.get("type", [])}
    _nodenorm_cache[curie] = res
    return res

def _load_node_ids(path=NODES_TSV):
    t0 = time.perf_counter()
    ids = set()
    p = subprocess.Popen(["cut", "-f8", path], stdout=subprocess.PIPE, text=True, bufsize=1 << 20)
    for line in p.stdout:
        ids.add(line.rstrip("\n"))
    p.wait()
    print("[node ids] loaded " + format(len(ids), ",") + " ids in " + format(time.perf_counter() - t0, ".1f") + "s")
    return ids

KG2_NODE_IDS = _load_node_ids()   # ~7s, ~11.3M ids; reused for every query this session

def expand_to_kg2(curie):
    """NodeNorm equivalence clique, intersected with local KG2 node ids (replaces 'all')."""
    clique = nodenorm_clique(curie)["equivalent_ids"]
    kept = [c for c in clique if c in KG2_NODE_IDS]
    if curie in KG2_NODE_IDS and curie not in kept:
        kept.append(curie)
    return kept, clique

[node ids] loaded 11,342,763 ids in 5.5s


In [4]:
def _ci_regex(term):
    parts = []
    for ch in term.strip():
        if ch.isalpha():
            parts.append("[" + ch.lower() + ch.upper() + "]")
        elif ch in '"]^\\':
            parts.append(".")
        else:
            parts.append("[" + ch + "]")
    return ".*" + "".join(parts) + ".*"

def _rank(name, src, term_lc):
    nl = name.lower()
    if nl == term_lc:
        return 0
    if nl.startswith(term_lc):
        return 1
    if term_lc in nl:
        return 2
    return 3

def _search_nodes(term, limit=25):
    """Fallback: Souffle match() regex over KG2 node names/synonyms."""
    pat = _ci_regex(term)
    program = "\n".join([
        ".decl node(id:symbol, name:symbol, category:symbol, syn:symbol)",
        '.input node(IO=file, filename="' + NODES_TSV + '", columns="7:10:15:13")',
        ".decl cand(id:symbol, name:symbol, category:symbol, src:symbol)",
        'cand(i, n, c, "name") :- node(i, n, c, _), match("' + pat + '", n).',
        'cand(i, n, c, "syn") :- node(i, n, c, s), match("' + pat + '", s).',
        ".output cand",
    ])
    df = run_souffle(program, "cand")
    if len(df) == 0:
        return pd.DataFrame(columns=["id", "name", "category"])
    df.columns = ["id", "name", "category", "src"]
    tl = term.strip().lower()
    df["rank"] = [_rank(n, s, tl) for n, s in zip(df["name"], df["src"])]
    df["nlen"] = df["name"].str.len()
    df = df.sort_values(["rank", "nlen"]).drop_duplicates("id").head(limit)
    out = df[["id", "name", "category"]].reset_index(drop=True)
    print("[resolve:" + term + "] match() kept " + str(len(out)) + " of top candidates")
    return out

def resolve_term(term, limit=10):
    """NameRes concept lookup; fall back to Souffle match() when NameRes is empty."""
    hits = nameres_lookup(term, limit=limit)
    if hits:
        print("[resolve:" + term + "] NameRes -> " + str(len(hits)) + " concept(s)")
        return pd.DataFrame(hits)[["id", "name", "category", "score", "clique_n"]]
    print("[resolve:" + term + "] NameRes empty -> Souffle match() fallback")
    return _search_nodes(term)

EXTRACT_SYSTEM = (
    "Extract from the biomedical question the named entities to look up and the relation(s) it implies.\n"
    'Return ONLY JSON: {"entities": [<entity name>, ...], "predicates": [<predicate>, ...]}.\n'
    "entities: the specific things named in the question (drugs, diseases, genes, ...); [] if none.\n"
    "predicates: choose the relevant relation(s) ONLY from this list, or [] if unconstrained: {predicates}."
)

def retrieve(query):
    system = EXTRACT_SYSTEM.replace("{predicates}", ", ".join(KG2_PREDICATES))
    t0 = time.perf_counter()
    spec = _json(llm(system, query, max_tokens=400))
    print("[llm:extract] " + format(time.perf_counter() - t0, ".1f") + "s -> entities=" + str(spec.get("entities")) + " predicates=" + str(spec.get("predicates")))
    entities = {}
    for term in spec.get("entities", []):
        entities[term] = resolve_term(term)
    return {"entities": entities, "predicates": spec.get("predicates", [])}

## 3. Select node IDs

In [5]:
def _bind_concepts(term, df, picks):
    """NodeNorm-expand each picked concept and intersect with KG2 (replaces 'all')."""
    bound = []
    for i in picks:
        cid = df.loc[i, "id"]
        kept, clique = expand_to_kg2(cid)
        label = str(df.loc[i, "name"])[:40]
        if kept:
            print("  '" + label + "' (" + cid + ") -> " + str(len(kept))
                  + " KG2 id(s) from " + str(len(clique)) + "-member clique")
        else:
            print("  '" + label + "' (" + cid + "): no KG2 overlap")
        bound.extend(kept)
    return list(dict.fromkeys(bound))   # de-dup, preserve order

def _pick_from_kg2(term):
    """Souffle match() fallback with manual pick, for terms NameRes/NodeNorm miss."""
    fb = _search_nodes(term)
    if len(fb) == 0:
        return []
    print("KG2 match() candidates for '" + term + "' (number, 'all', or blank):\n")
    for i, row in fb.iterrows():
        print("  [" + str(i) + "] " + str(row["name"])[:55] + "  " + str(row["category"]) + "  " + str(row["id"]))
    sel = input("\nNumber, 'all', or blank: ").strip().lower()
    if sel == "all":
        return fb["id"].tolist()
    if sel.isdigit() and int(sel) in fb.index:
        return [fb.loc[int(sel), "id"]]
    return []

def select(retrieval):
    chosen = {}
    for term, df in retrieval["entities"].items():
        if df is None or len(df) == 0:
            print("No matches for '" + term + "'.")
            chosen[term] = []
            continue
        # 1) human picks which concept(s) they mean (auto when only one)
        if len(df) == 1:
            picks = [df.index[0]]
        else:
            print("Concepts for '" + term + "' (number, comma-separated, or blank to skip):\n")
            for i, row in df.iterrows():
                extra = "  score=" + str(row["score"]) if "score" in df.columns else ""
                print("  [" + str(i) + "] " + str(row["name"])[:55] + "  " + str(row["category"]) + "  " + str(row["id"]) + extra)
            sel = input("\nNumber(s), or blank to skip: ").strip()
            picks = [int(s) for s in sel.replace(",", " ").split() if s.isdigit() and int(s) in df.index]
        # 2) NodeNorm expands each concept to its KG2 id set
        bound = _bind_concepts(term, df, picks)
        # 3) fallback: clique has no KG2 overlap -> Souffle match()
        if picks and not bound:
            print("  -> no KG2 overlap; Souffle match() fallback for '" + term + "'")
            bound = _pick_from_kg2(term)
        chosen[term] = bound
    return chosen

## 4. Write the .dl

In [6]:
DL_SYSTEM = (
    "You write Soufflé Datalog to answer a biomedical question over the RTX-KG2 graph.\n"
    "Every program MUST begin with exactly these two lines:\n"
    + EDGE_HEADER + "\n\n"
    'edge(subj, pred, obj, neg) is one KG edge: subj-[pred]->obj are CURIEs; use only neg="False".\n\n'
    "You are given grounded bindings:\n"
    "- Each entity is a SET of equivalent CURIEs. Declare a membership relation and bind the slot "
    "to a variable ranging over the set, e.g.\n"
    "      .decl drug_set(x:symbol)\n"
    '      drug_set("ID1"). drug_set("ID2").\n'
    "  Use the EXACT CURIEs given.\n"
    "- Predicates are exact quoted values.\n\n"
    "Write as many rules as the question needs; multi-hop joins and recursion are allowed. "
    "The final answer relation MUST be named result and hold the CURIEs the question asks for.\n"
    "Output ONLY the program (no prose/markdown). End with .output result. "
    'Use lowercase relation names; avoid the built-in "cat".'
)

def write_dl(query, selection, predicates):
    lines = []
    for term, ids in selection.items():
        if ids:
            idset = ", ".join('"' + i + '"' for i in ids)
            lines.append("entity '" + term + "' = {" + idset + "}")
        else:
            lines.append("entity '" + term + "' = (unresolved)")
    lines.append("predicates = " + (", ".join(predicates) if predicates else "(none specified)"))
    user = "Question: " + query + "\n\nGrounded bindings:\n" + "\n".join(lines) + "\n\nWrite the Soufflé program."
    return _souffle_code(llm(DL_SYSTEM, user, max_tokens=900))

## Pipeline

In [7]:
query = input("Ask a question (e.g. 'what drugs treat parkinson disease?'): ").strip()

_pipe_t0 = time.perf_counter()
r = retrieve(query)
print("Predicates: " + str(r["predicates"]))
for term, df in r["entities"].items():
    print("\nEntity '" + term + "':")
    display(df)

selection = select(r)

_t = time.perf_counter()
program = write_dl(query, selection, r["predicates"])
print("[llm:write_dl] " + format(time.perf_counter() - _t, ".1f") + "s")
print("\n" + program + "\n")

result = run_souffle(program, "result")
if len(result) == 0:
    print("No results.")
    named = result
else:
    ids = result[0].astype(str).tolist()
    namer = "\n".join(
        [
            ".decl node(id:symbol, name:symbol, category:symbol)",
            '.input node(IO=file, filename="' + NODES_TSV + '", columns="7:10:15")',
            ".decl want(id:symbol)",
        ]
        + ['want("' + i + '").' for i in ids]
        + [
            ".decl named(id:symbol, name:symbol, category:symbol)",
            "named(i, n, c) :- node(i, n, c), want(i).",
            ".output named",
        ]
    )
    named = run_souffle(namer, "named")
    if len(named) > 0:
        named.columns = ["id", "name", "category"]
    print(str(len(named)) + " result(s):")
print("[pipeline total] " + format(time.perf_counter() - _pipe_t0, ".1f") + "s")
display(named)

[llm:extract] 2.7s -> entities=['parkinson disease'] predicates=['biolink:treats']
[resolve:parkinson disease] NameRes -> 10 concept(s)
Predicates: ['biolink:treats']

Entity 'parkinson disease':


,id,name,category,score,clique_n
0,MONDO:0005180,Parkinson disease,biolink:Disease,3588.5,16
1,MONDO:0006966,secondary Parkinson disease,biolink:Disease,1090.9,12
2,MONDO:0001945,postencephalitic Parkinson disease,biolink:Disease,1056.8,11
3,MONDO:0008199,late-onset Parkinson disease,biolink:Disease,875.2,8
4,MONDO:0008200,autosomal dominant Parkinson disease 1,biolink:Disease,862.6,9
5,MONDO:0013060,autosomal recessive Parkinson disease 14,biolink:Disease,862.6,9
6,PANTHER.PATHWAY:P00049,Parkinson disease,biolink:Pathway,855.3,1
7,MONDO:0013625,Parkinson disease 17,biolink:Disease,827.6,6
8,MONDO:0011562,autosomal dominant Parkinson disease 4,biolink:Disease,779.0,7
9,MONDO:0014231,juvenile onset Parkinson disease 19A,biolink:Disease,779.0,7


Concepts for 'parkinson disease' (number, comma-separated, or blank to skip):

  [0] Parkinson disease  biolink:Disease  MONDO:0005180  score=3588.5
  [1] secondary Parkinson disease  biolink:Disease  MONDO:0006966  score=1090.9
  [2] postencephalitic Parkinson disease  biolink:Disease  MONDO:0001945  score=1056.8
  [3] late-onset Parkinson disease  biolink:Disease  MONDO:0008199  score=875.2
  [4] autosomal dominant Parkinson disease 1  biolink:Disease  MONDO:0008200  score=862.6
  [5] autosomal recessive Parkinson disease 14  biolink:Disease  MONDO:0013060  score=862.6
  [6] Parkinson disease  biolink:Pathway  PANTHER.PATHWAY:P00049  score=855.3
  [7] Parkinson disease 17  biolink:Disease  MONDO:0013625  score=827.6
  [8] autosomal dominant Parkinson disease 4  biolink:Disease  MONDO:0011562  score=779.0
  [9] juvenile onset Parkinson disease 19A  biolink:Disease  MONDO:0014231  score=779.0
  'Parkinson disease' (MONDO:0005180) -> 6 KG2 id(s) from 16-member clique
[llm:write_dl] 4.7s

,id,name,category
0,UMLS:C0003779,argipressin,biolink:NamedThing
1,UMLS:C0011701,desmopressin,biolink:NamedThing
2,UMLS:C0037125,Silver,biolink:SmallMolecule
3,UMLS:C0023870,lithium,biolink:ChemicalEntity
4,UMLS:C0009170,cocaine,biolink:ChemicalEntity
...,...,...,...
1622,DrugCentral:3521,rasagiline mesylate,biolink:ChemicalEntity
1623,DrugCentral:2697,RO 40-7592,biolink:ChemicalEntity
1624,DrugCentral:5143,BIA 9-1067,biolink:ChemicalEntity
1625,DrugCentral:4882,KW-6002,biolink:ChemicalEntity
